
## 📘 1. Business Understanding

**Objective**: Build a summarization model that helps users digest long AI/ML articles quickly.

**Why?** There's an overwhelming amount of AI/ML content (blogs, papers, newsletters), and professionals or students don’t always have time to read it all. This tool helps by summarizing that content into short, readable, and non-jargon-heavy blurbs.

**Target users**: Data scientists, ML practitioners, students, and e-learners.

**Goal**: Provide useful summaries that are easier and quicker to read than the original content — while still accurate and useful. The model will be evaluated based on quality improvements over a baseline model using standard NLP metrics (ROUGE, BERTScore).


In [ ]:
# 📘 Notebook: Fine-tuning DistilBART with Chunking and BERTScore + ROUGE Evaluation  

### 🔍 Code Explanation
Evaluates the model using ROUGE and BERTScore metrics.

- ROUGE measures n-gram overlap (surface similarity).
- BERTScore measures semantic similarity using contextual embeddings.

These metrics show how close the model's output is to the GPT-4o-generated reference summaries.

In [2]:
# 🧩 1. Install Required Libraries
#!pip install -q transformers evaluate datasets bert-score sentencepiece rouge-score
# %pip uninstall -y transformers
# %pip install transformers==4.54.0 accelerate datasets evaluate bert-score sentencepiece rouge_score
#stable version
# %pip install transformers==4.38.2 accelerate datasets evaluate bert-score sentencepiece rouge_score

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.3 MB/s eta 0:00:00:00:0100:01
   ━━━━

In [21]:
# 🧪 2. Import Libraries
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate
from tqdm import tqdm
tqdm.pandas()

import os
# os.environ["WANDB_DISABLED"] = "true"

### 🔍 Code Explanation
This loads the pre-trained `distilbart-cnn-12-6` model and its tokenizer. They will be fine-tuned on our dataset of AI/ML content.


## 📗 2. Data Understanding

**Data Source**:
- AI/ML Blogs: Google AI Blog, Hugging Face Blog, Dev.to
- Research Abstracts: arXiv (AI category)

**What’s in the dataset**:
- `title`: Title of article
- `content`: Full text scraped from source
- `labeled_summary`: Summary generated using GPT-4o (short, human-readable)

**Why use GPT-4o summaries?**
They act as the “ground truth” summaries for the model to learn from. They're generally accurate and are also easier to read than extractive summaries. The generated summaries will also follow the set criteria we have for what makes a good summary

**Volume**: We are using 4000 records of data to train the model


In [61]:
# 📊 3. Load the Dataset
df = pd.read_csv("/kaggle/input/3cols-final-summary-dataset-gpt4o/fixed_final_summaries_4o.csv")
df = df.dropna(subset=["content", "summary"])

### 🔍 Code Explanation
This loads the AI/ML article dataset from a CSV file into a pandas DataFrame. It contains `title`, `content`, and GPT-4o-generated `labeled_summary` columns.

In [62]:
# 🧠 4. Initialize Tokenizer and Model
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [63]:
# 🔪 5. Chunking Function (to handle long input content)
MAX_INPUT_TOKENS = 1024
CHUNK_OVERLAP = 50

In [64]:
def chunk_text(text, tokenizer, max_length=MAX_INPUT_TOKENS, overlap=CHUNK_OVERLAP):
    tokens = tokenizer.encode(text, truncation=False)
    chunks = []
    i = 0
    while i < len(tokens):
        chunk = tokens[i:i + max_length]
        decoded_chunk = tokenizer.decode(chunk, skip_special_tokens=True)
        chunks.append(decoded_chunk)
        i += max_length - overlap
    return chunks

### 🔍 Code Explanation: `chunk_text()` Function — Handling Long Inputs

Transformer models like **DistilBART** have a maximum input size of **1024 tokens**. If an article exceeds this limit, the excess text will be truncated and lost — potentially omitting important information.

The `chunk_text()` function solves this problem by splitting long texts into smaller overlapping chunks that fit within the model’s token limit.

---

#### 🧠 How It Works:
- **`tokenizer.encode(text, truncation=False)`**:
  - Converts the input text into token IDs without cutting off extra content.
- **While loop**:
  - Iterates over the tokens in sliding-window fashion.
  - Each iteration extracts a chunk of `max_length` tokens (default = 1024).
- **`overlap`** (default = 50 tokens):
  - Ensures that each chunk overlaps with the previous one.
  - This preserves context across chunk boundaries and improves summary continuity.
- **`tokenizer.decode(..., skip_special_tokens=True)`**:
  - Converts each chunk back into text format (still clean, no special tokens).
- **Returns**:
  - A list of chunked text strings, each suitable for input into the summarization model.

---

#### ✅ Why This Is Important:
- Prevents input truncation errors in long articles or papers.
- Improves summary quality by maintaining contextual flow between chunks.
- Enables the model to summarize content that exceeds the model’s native token limit by processing chunks separately.


In [65]:
# Apply chunking to content
df["chunked_inputs"] = df["content"].progress_apply(
    lambda x: chunk_text(x, tokenizer)
)

100%|██████████| 4035/4035 [00:07<00:00, 533.19it/s] 


### 🔍 Code Explanation: Chunk Flattening & Preprocessing

This block prepares the dataset for training by performing two key steps:

---

#### 🧱 1. Flattening Chunked Data

Each article may be too long for the DistilBART model (which has a 1024-token input limit). To handle this, we previously split the article into overlapping **chunks** using the `chunk_text()` function.

In this section:

- We iterate over each row in the original DataFrame (`df`)
- For each row, we loop through the list of `chunked_inputs` (individual text chunks)
- Each chunk is paired with the **same summary** from the original row (`row["summary"]`)
- These chunk-summary pairs are stored in a new list `flat_data`
- This list is converted into a new DataFrame and then a Hugging Face `Dataset`

✅ **Why this matters**: The model expects one input-output pair per row. Flattening ensures each chunk becomes a separate training example — enabling the model to learn from all parts of long articles.

---

#### ⚙️ 2. Preprocessing Function

The `preprocess()` function prepares the input data for the model by tokenizing and formatting it correctly.

Here's what it does:

- **Tokenizes** the `input` (a chunk of article text)
  - Truncates to `MAX_INPUT_TOKENS` if it's too long
  - Pads if it's too short
- **Tokenizes** the `summary` (the GPT-4o-labeled target text)
  - Truncates and pads to a maximum length of `128` tokens
- **Adds labels**: It sets `inputs["labels"] = targets["input_ids"]`
  - This tells the model what output sequence to learn to generate during training

It returns a dictionary with:
- `input_ids`: Encoded input tokens
- `attention_mask`: Indicates which tokens are actual content vs. padding
- `labels`: Encoded target summary tokens

✅ **Why this matters**: Transformers need fixed-size tokenized inputs and target labels. This function ensures the data is ready for use with the Hugging Face `Trainer` class.


In [66]:
# Flatten chunked data
flat_data = []
for _, row in df.iterrows():
    for chunk in row["chunked_inputs"]:
        flat_data.append({"input": chunk, "summary": row["summary"]})

In [67]:
flat_df = pd.DataFrame(flat_data)
hf_dataset = Dataset.from_pandas(flat_df)

In [68]:
# 🧪 Preprocessing Function
MAX_TARGET_LENGTH = 128
def preprocess(batch):
    inputs = tokenizer(batch["input"], truncation=True, padding="max_length", max_length=MAX_INPUT_TOKENS)
    targets = tokenizer(batch["summary"], truncation=True, padding="max_length", max_length=MAX_TARGET_LENGTH)
    inputs["labels"] = targets["input_ids"]
    return inputs

In [69]:
tokenized_ds = hf_dataset.map(preprocess, batched=True, remove_columns=hf_dataset.column_names)

Map:   0%|          | 0/5260 [00:00<?, ? examples/s]

In [ ]:
# 🏋️ Training Arguments

training_args = Seq2SeqTrainingArguments(
    output_dir="outputs",
    # evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

### 🔍 Code Explanation: Seq2SeqTrainingArguments
Defines training hyperparameters like learning rate, batch size, and evaluation strategy. These control how the model learns during fine-tuning using the Hugging Face `Trainer` API.


## 📕 4. Modeling

We use the `distilbart-cnn-12-6` model from Hugging Face — a smaller version of BART trained for summarization.

**Model type**: Encoder-Decoder (Seq2Seq) for text generation.

**Training strategy**:
- Fine-tune on our blog dataset using labeled summaries from GPT-4o
- Use Hugging Face `Trainer` API
- Hyperparameters: batch size 8–16, learning rate 2e-5, 3–5 epochs


In [72]:
# 🤖 Setup Trainer
data_collator = DataCollatorForSeq2Seq(tokenizer)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    eval_dataset=tokenized_ds.select(range(100)), #was previously 100
    # tokenizer=tokenizer,
    data_collator=data_collator,
)


### 🔍 Code Explanation: Seq2SeqTrainer

The `Seq2SeqTrainer` is a Hugging Face class designed for training encoder-decoder models like DistilBART for tasks such as summarization or translation.

Here's what each part of the configuration does:

- **model**: The transformer model we are fine-tuning
- **args**: Training arguments like batch size, learning rate, evaluation strategy, epochs
- **train_dataset**: The input data used for training the model
- **eval_dataset**: A separate split used to monitor validation performance
- **tokenizer**: Used to encode the inputs and decode the outputs for training
- **data_collator**: Helps batch inputs of different lengths and pad them properly
- **compute_metrics**: A custom function to compute ROUGE and BERTScore during evaluation

This trainer handles the training loop, validation, checkpointing, and evaluation automatically.


In [73]:
# 🚀 Train the Fine-tuned Model
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,2.299100
1000,1.833500
1500,1.629500


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarnin

TrainOutput(global_step=1974, training_loss=1.810921310533023, metrics={'train_runtime': 4244.6498, 'train_samples_per_second': 3.718, 'train_steps_per_second': 0.465, 'total_flos': 2.442605270925312e+16, 'train_loss': 1.810921310533023, 'epoch': 3.0})

### 📉 Training Loss
Training loss **consistently decreased** during fine-tuning:

| Step | Training Loss |
|------|----------------|
| 500  | 2.2991         |
| 1000 | 1.8335         |
| 1500 | 1.6295         |

This indicates that the model effectively learned from the GPT-4o-labeled summaries and adapted to the domain-specific AI/ML content.

### 🔍 Code Explanation
This line starts the actual fine-tuning of the model on the training dataset. The model learns to generate summaries that match the GPT-4o labels.

In [74]:
# 🔍 Load Evaluation Metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [ ]:
# 📏 Helper Function to Evaluate Any Model

def evaluate_model(model, tokenizer, dataset):
    eval_trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        args=training_args,
    )
    predictions = eval_trainer.predict(dataset)
    decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)

    # Compute ROUGE
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # Compute BERTScore
    bert_result = bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="en")

    bert_avg = {
        "precision": round(sum(bert_result["precision"]) / len(bert_result["precision"]), 4),
        "recall": round(sum(bert_result["recall"]) / len(bert_result["recall"]), 4),
        "f1": round(sum(bert_result["f1"]) / len(bert_result["f1"]), 4)
    }

    return rouge_result, bert_avg, decoded_preds, decoded_labels


## 📒 5. Evaluation

After training, we compare the model’s summaries against the GPT-4o summaries using:

### 🔹 ROUGE (Recall-Oriented Understudy for Gisting Evaluation)
- **ROUGE-1**: Overlap of single words (unigrams)
- **ROUGE-2**: Overlap of word pairs (bigrams)
- **ROUGE-L**: Longest common subsequence — captures fluency and sentence structure
- **ROUGE-Lsum**: Sentence-level version of ROUGE-L for summaries

Higher ROUGE scores suggest better overlap with the human (GPT-4o) summaries.

### 🔹 BERTScore
Instead of just checking for matching words, BERTScore looks at **semantic similarity** using a BERT-based model.
- **Precision**: How much of the generated summary is semantically similar to the reference
- **Recall**: How much of the reference summary is captured in the generated one
- **F1**: Harmonic mean of the two

This is useful for **abstractive summarization**, where wording differs but meaning can still match.


In [76]:
# 🎯 Evaluate Baseline (Unfine-tuned)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
baseline_rouge, baseline_bert, baseline_summaries, references = evaluate_model(baseline_model, tokenizer, tokenized_ds.select(range(100)))#tokenizer_ds.select(range(100))
print("\n📊 Baseline ROUGE:", baseline_rouge)
print("📊 Baseline BERTScore:", baseline_bert)


/tmp/ipykernel_36/1803282067.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  eval_trainer = Seq2SeqTrainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



📊 Baseline ROUGE: {'rouge1': 0.29208831162610704, 'rouge2': 0.06298202050562325, 'rougeL': 0.17105664683138228, 'rougeLsum': 0.2009150160462877}
📊 Baseline BERTScore: {'precision': 0.85, 'recall': 0.8474, 'f1': 0.8487}


### 📊 Baseline Model Evaluation
These are the ROUGE and BERTScore metrics for the **pre-trained DistilBART model** without any fine-tuning. They serve as a performance benchmark to compare our fine-tuned model against.

In [77]:
# 🎯 Evaluate Finetuned
finetuned_rouge, finetuned_bert, finetuned_summaries, _ = evaluate_model(model, tokenizer, tokenized_ds.select(range(100))) #range was 100 before
print("\n📊 Finetuned ROUGE:", finetuned_rouge)
print("📊 Finetuned BERTScore:", finetuned_bert)




/tmp/ipykernel_36/1803282067.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  eval_trainer = Seq2SeqTrainer(



📊 Finetuned ROUGE: {'rouge1': 0.5015079748985711, 'rouge2': 0.2009015794990769, 'rougeL': 0.30756104735837697, 'rougeLsum': 0.37530642725495517}
📊 Finetuned BERTScore: {'precision': 0.8921, 'recall': 0.8915, 'f1': 0.8917}


The metric benchmarks that we are aiming for are:

| Metric         | Acceptable | Good      | Excellent |
| -------------- | ---------- | --------- | --------- |
| **ROUGE-1**    | 0.30–0.35  | 0.36–0.45 | > 0.45    |
| **ROUGE-2**    | 0.08–0.12  | 0.13–0.20 | > 0.20    |
| **ROUGE-L**    | 0.25–0.35  | 0.36–0.42 | > 0.42    |
| **ROUGE-Lsum** | 0.28–0.36  | 0.37–0.45 | > 0.45    |

### 📈 Metric Improvements:
| Metric        | Baseline | Fine-Tuned | 
|---------------|----------|------------|
| ROUGE-1       | 0.2921   | 0.5015    | 
| ROUGE-2       | 0.063   | 0.2009    |
| ROUGE-L       | 0.1711   | 0.3076    | 
| ROUGE-Lsum    | 0.2009| 0.3753 |
| BERTScore-F1  | 0.8487 | 0.8917 |


### 📊 Code Explanation: Comparing Summaries Side-by-Side

This DataFrame is created to **visually compare how different models summarize the same input**. It includes the first 100 samples from the test set and displays:

- **Input**: The original chunked text (typically a paragraph from a blog or article)
- **Reference**: The target summary generated earlier by GPT-4o — treated as the gold standard
- **Baseline Summary**: The summary generated by the **pre-trained DistilBART** model (before fine-tuning)
- **Finetuned Summary**: The summary generated by the **fine-tuned DistilBART** model (after training on your domain-specific GPT-4o data)

---

#### ✅ Why this is useful:

- Helps **qualitatively evaluate** how well the fine-tuned model performs compared to the baseline.
- Lets you identify examples where:
  - The baseline summary is generic or off-topic.
  - The fine-tuned model produces a more relevant or clearer summary.
- Useful for demos, visual inspection, and documenting model improvements in presentations or reports.


In [78]:
# 📝 Compare Summaries Side-by-Side
comparison_df = pd.DataFrame({
    "Input": flat_df.loc[:99, "input"].values,
    "Reference": references,
    "Baseline Summary": baseline_summaries,
    "Finetuned Summary": finetuned_summaries
})

In [79]:
# Display a few examples
comparison_df.sample(5, random_state=42)

,Input,Reference,Baseline Summary,Finetuned Summary
83,"b model, which is much more reasonable and fit...",Preference optimization is a method used to tr...,Our memory calculation isn't exact as it does...,This article explains how to train a large lan...
53,MotivationBackground on LoRAMulti-LoRA Serving...,The article discusses a new method called Mult...,Multi-LoRA serving is a technique to fine-tun...,The article discusses a new feature called Mul...
70,How the Reformer uses less than 8GB of RAM to ...,The Reformer is a new model designed to handle...,How the Reformer uses less than 8GB of RAM to...,The Reformer model is a powerful tool for hand...
45,"UB device radix sort, a highly optimized sort ...",3D Gaussian Splatting is a method for creating...,UB device radix sort is a highly optimized sor...,Hugging Face has introduced a new way to speed...
44,What is 3D Gaussian Splatting?How it works1. S...,3D Gaussian Splatting is a method for creating...,3D Gaussian Splatting allows real-time render...,3D Gaussian Splatting is a new method used to ...


In [ ]:
comparison_df.to_csv('/kaggle/working/comparison_df')


## ✅ Conclusion

Fine-tuning the `distilbart-cnn-12-6` model on GPT-4o-labeled AI/ML blog summaries led to **significant performance improvements** across all evaluation metrics.

### 📈 Metric Improvements:
| Metric        | Baseline | Fine-Tuned | % Improvement |
|---------------|----------|------------|----------------|
| ROUGE-1       | 0.2921   | 0.5015    | **71.69%** |
| ROUGE-2       | 0.063   | 0.2009    | **218.89%** |
| ROUGE-L       | 0.1711   | 0.3076    | **79.78%** |
| ROUGE-Lsum    | 0.2009| 0.3753 | **86.81%** |
| BERTScore-F1  | 0.8487 | 0.8917 | **5.07%** |

These improvements show that the model:
- Captures more relevant keywords and sentence structure (ROUGE)
- Generates more semantically aligned summaries (BERTScore)






## 💡 Recommendations

To build on this successful fine-tuning effort, consider the following next steps:

1).  Collect human ratings in the app to benchmark perceived quality and drive active‑learning loops.

2).  Explore longer‑context models or hierarchical chunking to reduce truncation effects and push ROUGE‑L toward Good.

3).  Calibrate generation settings (length penalties, beams) per content type (blog vs abstract) for optimal readability.

4).  Expand and diversify training data (methods/experiments sections, different venues) to increase robustness.

